#  Single-Agent Smart Assistant Pipeline

Build a Single-Agent Smart Assistant that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

**Handles:**
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

**Bonus added:** logging, a 3rd tool (word counter), and regex-based
expression extraction so the calculator never crashes on plain sentences.

In [1]:
import re
import logging

# Logging setup (BONUS): gives us visibility into every routing decision
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("agent")

##  Tool 1: Calculator

In [2]:
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        # eval() is safe here because we ONLY ever pass it a pre-cleaned
        # string of digits/operators (see extract_expression below) —
        # never the raw user query.
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

##  Tool 2: Keyword Extractor

In [3]:
def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        # BUG FIX: original stub had no return here, which meant a failure
        # would silently return None instead of a list. Fixed to return [].
        return []

##  Tool 3 (BONUS): Word Counter\n\nA small extra tool showing the pipeline is easy to extend. Triggered by the word `"count"`.

In [4]:
def word_counter(text: str) -> dict:
    """Count words and characters in a piece of text."""
    try:
        words = text.split()
        return {"word_count": len(words), "char_count": len(text)}
    except Exception:
        return {"word_count": 0, "char_count": 0}

## Helper functions used by the agent for routing\n\n`extract_expression` pulls just the math out of a sentence like *'Calculate 20 + 5'* → *'20 + 5'*, since you can't `eval()` a whole sentence.\n\n`extract_keyword_source` strips instruction words like *'extract keywords from'* so the tool only processes the actual content.

In [5]:
def extract_expression(text: str):
    """Pull out just the mathematical part of a query."""
    match = re.search(r"[\d\s\+\-\*/\.\(\)]+", text)
    if match:
        cleaned = match.group().strip()
        return cleaned if cleaned else None
    return None


def extract_keyword_source(text: str) -> str:
    """If query looks like 'Extract keywords from <sentence>', only
    run keyword extraction on <sentence>, not the instruction words."""
    lower = text.lower()
    marker = "keywords from"
    if marker in lower:
        idx = lower.index(marker) + len(marker)
        return text[idx:].strip()
    return text

##  Agent Function\n\nUnderstands the query, routes it to the right tool, and always returns structured JSON: `{\"type\": ..., \"result\": ...}` — even on failure.

In [6]:
def agent(query: str) -> dict:
    """Routes the query and always returns structured JSON."""

    # --- Basic input validation / error handling ---
    if not isinstance(query, str) or not query.strip():
        logger.warning("Empty or invalid query received")
        return {"type": "error", "result": "Query must be a non-empty string"}

    query_lower = query.lower()
    logger.info(f"Received query: {query!r}")

    try:
        # --- Route 1: Math -----------------------------------
        if "calculate" in query_lower:
            expression = extract_expression(query)
            if not expression:
                logger.warning("No numeric expression found in calc query")
                return {"type": "error", "result": "No valid expression found in query"}

            calc_result = calculator(expression)
            if calc_result == "Error in calculation":
                logger.warning(f"Calculator failed on expression: {expression!r}")
                return {"type": "error", "result": calc_result}

            logger.info(f"Routed to calculator -> {calc_result}")
            return {"type": "calculation", "result": calc_result}

        # --- Route 2: Keywords ---------------------------------
        elif "keyword" in query_lower:  # catches "keyword" and "keywords"
            source_text = extract_keyword_source(query)
            keywords = extract_keywords(source_text)
            logger.info(f"Routed to keyword extractor -> {keywords}")
            return {"type": "keywords", "result": keywords}

        # --- Route 3 (BONUS): Word count -----------------------
        elif "count" in query_lower:
            stats = word_counter(query)
            logger.info(f"Routed to word counter -> {stats}")
            return {"type": "count", "result": stats}

        # --- Route 4: General / fallback ------------------------
        else:
            logger.info("Routed to general response (no tool matched)")
            return {
                "type": "general",
                "result": f"I received your query: '{query}'. "
                          f"No specific tool matched, so this is a general response."
            }

    except Exception as e:
        # Catch-all so the agent NEVER crashes, it always returns JSON
        logger.error(f"Unexpected error while handling query: {e}")
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}

##  Test Cases

In [7]:
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Calculate ten plus five",              # should error gracefully (no digits)
    "Count words in this sentence please",  # bonus tool
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['artificial', 'industries', 'transforming', 'intelligence']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I received your query: 'What is machine learning?'. No specific tool matched, so this is a general response."}
--------------------------------------------------
Query: Calculate ten plus five
Response: {'type': 'error', 'result': 'No valid expression found in query'}
--------------------------------------------------
Query: Count words in this sentence please
Response: {'type': 'count', 'result': {'word_count': 6, 'char_count': 35}}
--------------------------------------------------


##  Interactive Mode\n\nRun this cell and type queries. Type `exit` to stop.

In [8]:
while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop): exit
